# 03 — QSFP interconnect: link up, MTU, name the right port

**Milestone (L1–L2):** the **high-speed** cable between Sparks is seated, the kernel sees the link **UP**, and you know which **`enp…` / `roce…`** names map to **cluster** vs **router** Ethernet.

Retail QSFP DAC/AOC is often sold as **“200G”** — that is usually **200 gigabit-class** (Gb/s), not 200 **gigabytes** per second. Still vastly faster than trying to carry NCCL over a mistake on the wrong 1 Gb port.

**Green signal:** `ibdev2netdev` / `ip link` show the **cluster** interface **UP** (not necessarily the same as “I plugged in Cat6”).

## Checklist

- [ ] Cable type matches **NVIDIA playbook** (correct port pairing).
- [ ] Run on **each** Spark (SSH session): `ip -br link` and note **UP** interfaces.
- [ ] Install/use **`ibdev2netdev`** if available — maps IB/RoCE device to Linux netdev.
- [ ] Set **MTU** per playbook (often 9000+ on the cluster-facing iface where appropriate).
- [ ] Write down **`ETHERNET_IF`** (e.g. `enp1s0f0np0`) and **`IB_IF` / RoCE** name (`rocep1s0f0`) for later `launch-cluster.sh` / NCCL.

If autodiscovery in scripts **hangs for minutes**, wrong iface is a top suspect.

In [ ]:
# Run this cell ON THE SPARK (head) after SSH — not on laptop
import subprocess

def run(cmd: list[str]) -> str:
    r = subprocess.run(cmd, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print("=== ip -br link ===")
print(run(["ip", "-br", "link"]))

print("\n=== ibdev2netdev (if installed) ===")
print(run(["bash", "-lc", "command -v ibdev2netdev >/dev/null && ibdev2netdev || echo 'install rdma-core / perftest tools'"]))

## Gate

If the **QSFP** link is not **UP**, stop: no `10.0.0.x` work ([`04`](04_l3_cluster_subnet_ping_and_routes.ipynb)) will save you.

**Next:** assign **cluster subnet** IPs on the interfaces you just identified.